# Explore Olist Dataset

Inspecting each table: columns, size, missing values.

## Orders

`olist_orders_dataset.csv`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
orders.shape

(99441, 8)

99441 rows, 8 columns. Close to the 100k orders claimed on Kaggle. Row 1 of the file was the header, so the CSV has 99442 lines.

In [2]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


One row per order. Columns cover the lifecycle: purchase, payment approval, hand-off to carrier, arrival, plus the estimated delivery promised at checkout. `order_id` looks like the primary key.

In [3]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Every column is `text`, including the five date columns. pandas did not parse dates automatically, so date math has to wait until they are converted in the cleaning step.

In [4]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing values sit in the three pipeline timestamps, not in keys or purchase time. `order_approved_at`: 160 null. Carrier: 1783 null. Delivery to customer: 2965 null. `order_estimated_delivery_date` is complete, which makes sense because the estimate exists at checkout.

In [5]:
orders["order_status"].value_counts().sort_index()

order_status
approved           2
canceled         625
created            5
delivered      96478
invoiced         314
processing       301
shipped         1107
unavailable      609
Name: count, dtype: int64

8 statuses. 96478 of 99441 orders (about 97%) are `delivered`. The rest are early stages (`created`, `approved`, `processing`, `invoiced`), `shipped` but not arrived (1107), or failed (`canceled`, `unavailable`).

In [6]:
orders.groupby("order_status")[[
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]].apply(lambda d: d.isnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


Nulls mostly line up with status, so they look structural rather than random. For example: all 1107 `shipped` orders have no delivery date yet, which is expected. One anomaly: 8 rows are `delivered` but still missing the delivery timestamp.

In [7]:
weird = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].isnull())
]
weird

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


The 8 delivered orders with no arrival time. 7 of them have every earlier date, only one `order_delivered_carrier_date` timestamp is blank. Looks like a recording gap, not lost packages. Too few to move any metric. Exclude them from delay calculations instead of dropping the rows.

## Order items

`olist_order_items_dataset.csv`

In [8]:
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
items.shape

(112650, 7)

112650 rows, 7 columns. More rows than orders (112650 vs 99441), so some orders have multiple items.

In [9]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Columns cover the order specifics: order, item, product and seller ID, shipping deadline date, order cost and delivery cost. `order_item_id` is the item's position in the order (1 = first item), that is why one `order_id` can appear on several rows.

In [10]:
items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Order, product and seller IDs and the shipping deadline are `str`. `order_item_id` is `int64` because it is the item number within the order. Price and freight are `float64` because they are currency values.

In [11]:
items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

No missing values.

In [21]:
items.groupby("order_id").size().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

Most orders have 1 item. The average is slightly above 1, and the largest order has 21 items.

In [22]:
orders_without_items = set(orders["order_id"]) - set(items["order_id"])
len(orders_without_items)

775

775 orders have no item rows at all.

In [23]:
orders[orders["order_id"].isin(orders_without_items)]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Almost all of the no item orders are unavailable (603) or canceled (164): orders that failed before items existed. A few are early-stage, plus 1 marked shipped which looks wrong.

In [17]:
items["price"].describe()

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Item prices range from 0.85 to 6735, with a median of 74.99. The average (120.65) sits above the median, so a few expensive items pull it up.

## Payments

`olist_order_payments_dataset.csv`

In [30]:
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
payments.shape

(103886, 5)

103886 rows, 5 columns. More rows than orders (103886 vs 99441), so some orders have more than one payment row.

In [31]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Columns cover the order payment data: order ID, payment method (credit card, debit card, etc.), amount paid, how many installments the payment was split into, and the sequence number of this payment within the order.

In [33]:
payments.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Order ID and payment type are `str`. Payment sequential and installments are `int64`, and payment_value is `float64` because it is a currency value.

In [34]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

No missing values.

In [49]:
counts = payments["payment_type"].value_counts()
shares = payments["payment_type"].value_counts(normalize=True).mul(100).round(1)
payment_table = pd.DataFrame({"count": counts, "percent": shares})
payment_table

,count,percent
payment_type,,
credit_card,76795,73.9
boleto,19784,19.0
voucher,5775,5.6
debit_card,1529,1.5
not_defined,3,0.0


About 74% of payments are made using a credit card. Boleto (bank slip, transfer after order) is second at 19%, then voucher and a small amount of debit card. Only 3 rows are not_defined, negligible.

In [66]:
per_order = payments.groupby("order_id")["payment_sequential"].max()

payment_table = pd.DataFrame({
    "count": per_order.value_counts().sort_index(),
    "percent": per_order.value_counts(normalize=True).mul(100).round(1),
})
payment_table

,count,percent
payment_sequential,,
1,96401,96.9
2,2458,2.5
3,303,0.3
4,108,0.1
5,52,0.1
6,36,0.0
7,28,0.0
8,11,0.0
9,9,0.0


About 97% of orders have a single payment. 2.5% use two payments, 0.3% use three. A few orders go much higher, up to 29 payments.

In [73]:
max_seq = payments.groupby("order_id")["payment_sequential"].max()
big = max_seq[max_seq >= 10].index

payments[payments["order_id"] == big[-1]].sort_values("payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
9665,fedcd9f7ccdc8cba3a18defedd1a5547,1,credit_card,1,1.67
27087,fedcd9f7ccdc8cba3a18defedd1a5547,2,voucher,1,7.76
87783,fedcd9f7ccdc8cba3a18defedd1a5547,3,voucher,1,26.94
45197,fedcd9f7ccdc8cba3a18defedd1a5547,4,voucher,1,10.33
36869,fedcd9f7ccdc8cba3a18defedd1a5547,5,voucher,1,9.76
103569,fedcd9f7ccdc8cba3a18defedd1a5547,6,voucher,1,8.60
95493,fedcd9f7ccdc8cba3a18defedd1a5547,7,voucher,1,11.78
15930,fedcd9f7ccdc8cba3a18defedd1a5547,8,voucher,1,31.43
14352,fedcd9f7ccdc8cba3a18defedd1a5547,9,voucher,1,5.30
44641,fedcd9f7ccdc8cba3a18defedd1a5547,10,voucher,1,13.27


Code: Take each order's highest payment_sequential to see how many payment rows it has. Orders with 10+ payments are rare; this pulls all payment rows for the last one in that list.

Result: One credit card payment followed by many small vouchers. The scattered index numbers are the CSV line numbers, so payments for one order are not stored next to each other.